In [ ]:
# Lets make sure we only use one GPU 
# before your neural/torch imports
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
import pandas as pd
import numpy as np
from neuralforecast import NeuralForecast
from neuralforecast.auto import AutoMLP
from neuralforecast.losses.pytorch import MAE
import optuna
from ray import tune
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from pathlib import Path

darts_colors = ["black", "#003DFD", "#b512b8", "#11a9ba", "#0d780f", "#f77f07", "#ba0f0f"]

class CFG:
    data_folder = Path.cwd().parent / "data"
    img_dim1 = 8
    img_dim2 = 4

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"PyTorch CUDA version: {torch.version.cuda}")

In [ ]:
# you can use this code to check if CUDA GPUs are installed
import subprocess
def check_nvidia_smi():
    try:
        output = subprocess.check_output(['nvidia-smi'])
        return output.decode('utf-8')
    except:
        return "nvidia-smi not found or not accessible"

print(check_nvidia_smi())

In [ ]:
# If GPU is not enabled you can install CUDA-enabled PyTorch directly with pip inside Poetry environment, if you have a GPU card that allows CUDA.
# make sure the cu### value relates to your version of cuda, newer versions of CUDA may be able to use older stable pytorch versions 
# poetry run pip install --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124 

In [ ]:
# We can use CPU cores instead. This is slower, but it will get there eventually. Good time to get a coffee when the models start processing, or do some additonal learning, documentation etc. 

import multiprocessing
import psutil

# Get CPU information
physical_cores = psutil.cpu_count(logical=False)  # Physical cores only
logical_cores = psutil.cpu_count(logical=True)    # Including virtual/logical cores

print(f"Physical CPU cores: {physical_cores}")
print(f"Logical CPU cores: {logical_cores}")

# Get CPU usage percentage
cpu_percent = psutil.cpu_percent(interval=1, percpu=True)
print(f"\nCPU Usage per core: {cpu_percent}")

# Get memory information
memory = psutil.virtual_memory()
print(f"\nTotal RAM: {memory.total / (1024 ** 3):.2f} GB")
print(f"Available RAM: {memory.available / (1024 ** 3):.2f} GB")
print(f"RAM Usage: {memory.percent}%")

In [ ]:
def load_m5_subset(data_folder: Path) -> pd.DataFrame:
    """
    Load and prepare the M5 subset data
    
    Parameters
    ----------
    data_folder : Path
        Path to the data directory
    
    Returns
    -------
    pd.DataFrame
        Processed DataFrame with proper date formatting and index
    """
    try:
        # Construct filepath
        filepath = data_folder / 'M5_t20_ABC.csv'
        
        # Verify file exists
        if not filepath.exists():
            raise FileNotFoundError(f"Data file not found at {filepath}")
            
        # Read data
        df = pd.read_csv(filepath, index_col=0)
        
        # Convert date to datetime
        df['date'] = pd.to_datetime(df['date'])
        
        # Remove last 28 days of each series
        df_sorted = df.sort_values(['item_id', 'date'])
        def remove_tail_values(group):
            return group.iloc[:-28]
        df = df_sorted.groupby('item_id').apply(remove_tail_values).reset_index(drop=True)
        
        # Add time-based features
        df['year'] = df['date'].dt.year
        df['month'] = df['date'].dt.month
        df['dayofweek'] = df['date'].dt.dayofweek
        df['quarter'] = df['date'].dt.quarter
        df['week_of_year'] = df['date'].dt.isocalendar().week
        
        return df
        
    except Exception as e:
        print(f"Error loading data: {str(e)}")
        raise

def analyze_dataset(df: pd.DataFrame) -> dict:
    """
    Analyze the dataset structure and provide summary statistics
    
    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame
    
    Returns
    -------
    dict
        Dictionary containing analysis results
    """
    # Get ABC distribution by unique products
    abc_dist = df.groupby('item_id')['ABC_class'].first().value_counts().to_dict()
    
    analysis = {
        'n_series': df['item_id'].nunique(),
        'n_stores': df['state_id'].nunique(),
        'date_range': (df['date'].min(), df['date'].max()),
        'total_sales': df['sold'].sum(),
        'mean_price': df['sell_price'].mean(),
        'abc_distribution': abc_dist,
        'sales_by_class': df.groupby('ABC_class')['sold'].sum().to_dict(),
        'avg_price_by_class': df.groupby('ABC_class')['sell_price'].mean().to_dict()
    }
    
    return analysis

def plot_sales_analysis(series: pd.DataFrame, figsize=(15, 10)):
    """
    Create a comprehensive sales analysis plot
    
    Parameters
    ----------
    series : pd.DataFrame
        Single time series data to analyze
    figsize : tuple
        Figure size for the plots
    """
    fig, axes = plt.subplots(2, 1, figsize=figsize)
    
    # Sales over time
    axes[0].plot(series['date'], series['sold'], color = darts_colors[1], alpha=0.7)
    axes[0].set_title(f"Sales Over Time for {series['item_id'].iloc[0]}")
    axes[0].set_xlabel('Date')
    axes[0].set_ylabel('Units Sold')
    
    # Sales by day of week
    sns.boxplot(data=series, x='dayofweek', y='sold', ax=axes[1], color = darts_colors[2])
    axes[1].set_title('Sales Distribution by Day of Week')
    axes[1].set_xlabel('Day of Week')
    axes[1].set_ylabel('Units Sold')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print("\nSummary Statistics:")
    print(series['sold'].describe())

In [ ]:
df = load_m5_subset(CFG.data_folder)

# Analyze dataset
analysis = analyze_dataset(df)
analysis

In [ ]:
# Select series to plot
series_to_plot = ['FOODS_3_090', 'FOODS_3_714', 'HOUSEHOLD_2_437', 'HOUSEHOLD_1_448']

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.tight_layout(pad=3.0)

# Flatten axes for easier iteration
axes = axes.flatten()

for idx, (series_id, ax) in enumerate(zip(series_to_plot, axes)):
    # Get single series
    series = df[df['item_id'] == series_id].copy()
    series = series.sort_values('date')
    
    # Get last 36 months
    last_date = series['date'].max()
    start_date = last_date - pd.DateOffset(months=36)
    series = series[series['date'] >= start_date]
    
    # Plot
    ax.plot(series['date'], series['sold'], 
            color=darts_colors[idx], 
            alpha=0.7)
    
    ax.set_title(f"Sales for {series_id}", fontsize=14)
    ax.set_xlabel('Date', fontsize=16)
    ax.set_ylabel('Units Sold', fontsize=16)
    ax.tick_params(axis='x', rotation=45)
plt.subplots_adjust(hspace=0.4, wspace=0.2)
plt.show()

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df = df[['id', 'date', 'sold', 'sell_price', 'year', 'month', 
               'dayofweek', 'quarter', 'week_of_year']].copy()
df = df.rename(columns={'date': 'ds', 'id': 'unique_id', 'sold': 'y'})
df = df.sort_values(['unique_id', 'ds'])

In [ ]:
df.head()

In [ ]:
# limit number of models to build 
unique_ids = df['unique_id'].unique()[:3]
df = df[df['unique_id'].isin(unique_ids)].copy()

In [ ]:
# Create train/test split - last 60 days are test
test_days = 60
train_df = df.groupby('unique_id').apply(lambda x: x.iloc[:-test_days]).reset_index(drop=True)
test_df = df.groupby('unique_id').apply(lambda x: x.iloc[-test_days:]).reset_index(drop=True)

futr_df = test_df[['unique_id', 'ds', 'sell_price', 'year', 'month', 'dayofweek', 'quarter', 'week_of_year']]

In [ ]:
#Define search space for AutoMLP -optuna version
def config_mlp(trial):
    return {
        "num_layers": trial.suggest_int("num_layers", 1, 4),
        "hidden_size": trial.suggest_int("hidden_size", 32, 512),  # Changed parameter name from input_size
        "learning_rate": trial.suggest_loguniform("learning_rate", 1e-4, 1e-1),
        "input_size": 90,  # Fixed lookback window
        "max_steps": 1000,
        "batch_size": 128,
        "random_seed": 28395,
        "scaler_type": "robust",
        "futr_exog_list": ['sell_price', 'year', 'month', 'dayofweek', 'quarter', 'week_of_year'],
        "early_stop_patience_steps": 5,
        "val_check_steps": 50,
        'accelerator': 'gpu', 
        'devices': [0]
    }
model = AutoMLP(
    h=60,  # Forecast horizon
    loss=MAE(),
    config=config_mlp,
    search_alg=optuna.samplers.TPESampler(),
    backend='optuna',
    num_samples=50,
    #cpus=44, 
    gpus = 1     
)

In [ ]:
# Doesn't recognie HyperOptSearch in system 
# from ray.tune.search.hyperopt import HyperOptSearch
# import hyperopt
# print(hyperopt.__version__)
# def config_mlp(trial):
#     return {
#         "num_layers": tune.randint(1, 3),
#         "hidden_size": tune.randint(10, 150),
#         "learning_rate": tune.loguniform(1e-4, 1e-1),
#         "input_size": 60,
#         "max_steps": 300,
#         "batch_size": 128,
#         "random_seed": 28395,
#         "scaler_type": "robust",
#         "futr_exog_list": ['sell_price', 'year', 'month', 'dayofweek', 'quarter', 'week_of_year'],
#         "early_stop_patience_steps": 5,
#         "val_check_steps": 25
#     }

# model = AutoMLP(
#     h=60,  
#     loss=MAE(),
#     config=config_mlp,
#     search_alg=HyperOptSearch(),  # Ray's implementation of HyperOpt
#     backend='ray',
#     num_samples=10,
#     resources_per_trial={"cpu": 2}  # Each trial uses 2 CPUs
# )

In [ ]:
nf = NeuralForecast(
    models=[model],
    freq='D'  # Daily data
)

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING) # Disables training prints 
nf.fit(df=train_df, val_size=60)

In [ ]:
results = nf.models[0].results.trials_dataframe()
results.drop(columns='user_attrs_ALL_PARAMS')
print(results) 

In [ ]:
Y_hat_df = nf.predict(futr_df=test_df)
Y_hat_df = Y_hat_df.reset_index().reset_index()
Y_hat_df.head()

In [ ]:
# Calculate metrics per series
from utilsforecast.losses import mse, mae, rmse, mape

metrics_list = []
for uid in test_df['unique_id'].unique():
    actuals = test_df[test_df['unique_id']==uid]['y'].values
    preds = Y_hat_df[Y_hat_df.index.get_level_values('unique_id')==uid]['AutoMLP'].values
    
    metrics = {
        'unique_id': uid,
        'MAE': mae(actuals, preds),
        'MSE': mse(actuals, preds),
        'RMSE': rmse(actuals, preds),
        'MAPE': mape(actuals, preds)
    }
    metrics_list.append(metrics)

evaluation_df = pd.DataFrame(metrics_list)

# Overall metrics (excluding unique_id column)
# Per series metrics
print("Metrics by Series:")
metrics_by_series = evaluation_df.groupby('unique_id')[['MAE','MSE','RMSE','MAPE']].mean()
print(metrics_by_series)

# Overall metrics (calculated on all values)
all_actuals = test_df['y'].values
all_preds = Y_hat_df['AutoMLP'].values

overall_metrics = {
   'MAE': mae(all_actuals, all_preds),
   'MSE': mse(all_actuals, all_preds),
   'RMSE': rmse(all_actuals, all_preds),
   'MAPE': mape(all_actuals, all_preds)
}

print("\nOverall Metrics (across all values):")
for metric, value in overall_metrics.items():
   print(f"{metric}: {value:.4f}")


In [ ]:
from statsforecast import StatsForecast

In [ ]:
plotting_id = test_df.unique_id.unique()
StatsForecast.plot(test_df, Y_hat_df, models=["AutoMLP"], unique_ids=plotting_id, engine='matplotlib')

NeuralForecast's 'Auto' models, select the best model (lowest validation loss) automatically selected for prediction. We can verify this by checking...

In [ ]:
best_trial = nf.models[0].results.best_trial
print("Best trial number:", best_trial.number)

In [ ]:
# Get model parameters used for prediction
model_params = nf.models[0][n].model_
print("\nModel parameters used for prediction:", model_params)

In [ ]:
nf.models[0].model.parameters()

In [ ]:
nf.models[0]

In [ ]:
print(torch.cuda.is_available())  # Should return True
print(torch.cuda.device_count())  # Number of available GPUs
print(torch.cuda.get_device_name(0))  # Name of the first GPU

In [ ]:
CUDA_VISIBLE_DEVICES=0 jupyter notebook